# Báo Cáo Thực Nghiệm: Mạng Tích Chập 1D (1D CNN) Bằng TensorFlow / Keras
## Bài Toán: Dự Đoán Nguy Cơ Tiểu Đường (Biomedical Tabular Classification)
**Học phần:** Phát triển Hệ thống Thông minh (Intelligent System Development)  
**Nhiệm vụ:** Assignment 04 - Ánh xạ kiến trúc 1D CNN sang TensorFlow (`layers.Conv1D`, Keras API)  
**Dữ liệu:** `diabetes_prediction_dataset.csv` (100,000 mẫu lâm sàng)


In [1]:
import os
os.environ["MPLBACKEND"] = "Agg"
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses
import numpy as np
import pandas as pd
import time
import json
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers, models, optimizers, losses, metrics

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Phiên bản TensorFlow: {tf.__version__}")


Phiên bản TensorFlow: 2.21.0


### 1. Đọc & Tiền Xử Lý Dữ Liệu Đồng Bộ


In [2]:
DATA_PATH = "../../../Assigment 02/diabetes/data/diabetes_prediction_dataset.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "src/Assigment 02/diabetes/data/diabetes_prediction_dataset.csv"

df_raw = pd.read_csv(DATA_PATH)
df_dedup = df_raw.drop_duplicates().copy()

df_clean = df_dedup[df_dedup['gender'] != 'Other'].copy()
df_clean['gender'] = (df_clean['gender'] == 'Male').astype(int)

smoking_map = {'never': 0, 'No Info': 0, 'former': 1, 'not current': 1, 'current': 2, 'ever': 2}
df_clean['smoking_history'] = df_clean['smoking_history'].map(smoking_map).fillna(0).astype(int)

features = ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level']
target = 'diabetes'

X_data = df_clean[features].values.astype(np.float32)
y_data = df_clean[target].values.astype(np.float32)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.20, random_state=RANDOM_SEED, stratify=y_data
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# Keras Conv1D format: (batch_size, length=8, in_channels=1)
X_train_tf = X_train_scaled[:, :, np.newaxis]
X_test_tf = X_test_scaled[:, :, np.newaxis]

print(f"X_train_tf shape: {X_train_tf.shape}, y_train shape: {y_train.shape}")
print(f"X_test_tf shape : {X_test_tf.shape}, y_test shape : {y_test.shape}")


X_train_tf shape: (76902, 8, 1), y_train shape: (76902,)
X_test_tf shape : (19226, 8, 1), y_test shape : (19226,)


### 2. Định Nghĩa Kiến Trúc 1D CNN Bằng TensorFlow / Keras
Xây dựng mô hình với Sequential API chuẩn hóa:
- `layers.Conv1D(filters=16, kernel_size=3, activation='relu')`
- `layers.Conv1D(filters=16, kernel_size=3, activation='relu')`
- `layers.MaxPooling1D(pool_size=2)`
- `layers.Flatten()`
- `layers.Dense(8, activation='relu')`
- `layers.Dense(1, activation='sigmoid')`


In [3]:
model = models.Sequential([
    layers.Input(shape=(8, 1)),
    layers.Conv1D(filters=16, kernel_size=3, activation='relu'),
    layers.Conv1D(filters=16, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.003),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 6, 16)          │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 4, 16)          │           784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 2, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │           264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,121 (4.38 KB)

 Trainable params: 1,121 (4.38 KB)

 Non-trainable params: 0 (0.00 B)

### 3. Huấn Luyện Mô Hình Với TensorFlow


In [4]:
epochs = 12
batch_size = 256

start_time = time.time()
print("Bắt đầu huấn luyện mô hình 1D CNN TensorFlow...")

fit_history = model.fit(
    X_train_tf, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test_tf, y_test),
    verbose=1
)

train_time = time.time() - start_time
print(f"\nHuấn luyện TensorFlow hoàn tất trong: {train_time:.2f} giây!")


Bắt đầu huấn luyện mô hình 1D CNN TensorFlow...
Epoch 1/12


  1/301 ━━━━━━━━━━━━━━━━━━━━ 3:56 787ms/step - accuracy: 0.8203 - loss: 0.6372

 49/301 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9061 - loss: 0.3323    

100/301 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9219 - loss: 0.2466

148/301 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9334 - loss: 0.2057

196/301 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9395 - loss: 0.1841

... (100 outputs left out)

235/301 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9696 - loss: 0.0867

282/301 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9697 - loss: 0.0868

301/301 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9696 - loss: 0.0872 - val_accuracy: 0.9710 - val_loss: 0.0889



Huấn luyện TensorFlow hoàn tất trong: 5.72 giây!


### 4. Đánh Giá Toàn Diện & Trực Quan Hóa Kết Quả


In [5]:
y_prob = model.predict(X_test_tf).ravel()
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_prob)

print(f"=== KẾT QUẢ ĐÁNH GIÁ 1D CNN (TENSORFLOW) ===")
print(f"• Accuracy : {acc:.4f}")
print(f"• Precision: {prec:.4f}")
print(f"• Recall   : {rec:.4f}")
print(f"• F1-Score : {f1:.4f}")
print(f"• ROC-AUC  : {auc:.4f}")
print(f"• Thời gian: {train_time:.2f} giây")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, epochs + 1), fit_history.history["loss"], 'o-', color='#f59e0b', label='Train Loss')
axes[0].plot(range(1, epochs + 1), fit_history.history["val_loss"], 's--', color='#ef4444', label='Val Loss')
axes[0].set_title('TensorFlow: Đường Cong Hàm Mất Mát', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].legend()

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Oranges', ax=axes[1], cbar=False)
axes[1].set_title('TensorFlow: Ma Trận Nhầm Lẫn', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nhãn Dự Đoán')
axes[1].set_ylabel('Nhãn Thực Tế')

plt.tight_layout()
os.makedirs("../reports/figures", exist_ok=True)
plt.savefig("../reports/figures/03_diabetes_tensorflow_results.png", dpi=300)
plt.show()


  1/601 ━━━━━━━━━━━━━━━━━━━━ 23s 40ms/step

182/601 ━━━━━━━━━━━━━━━━━━━━ 0s 276us/step

372/601 ━━━━━━━━━━━━━━━━━━━━ 0s 270us/step

562/601 ━━━━━━━━━━━━━━━━━━━━ 0s 268us/step

601/601 ━━━━━━━━━━━━━━━━━━━━ 0s 338us/step


=== KẾT QUẢ ĐÁNH GIÁ 1D CNN (TENSORFLOW) ===
• Accuracy : 0.9710
• Precision: 0.9939
• Recall   : 0.6757
• F1-Score : 0.8045
• ROC-AUC  : 0.9733
• Thời gian: 5.72 giây


/var/folders/x0/279_kvcj3dvbd1fkpvt_8j7r0000gn/T/ipykernel_3164/3089091177.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5. Lưu Trữ Kết Quả Thực Nghiệm & Tổng Hợp Đối Chuẩn


In [6]:
results_df = pd.DataFrame([{
    "Framework": "TensorFlow",
    "Dataset": "Diabetes (100k)",
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1": f1,
    "ROC_AUC": auc,
    "Train_Time_Sec": train_time,
    "Epochs": epochs,
    "Batch_Size": batch_size
}])

results_df.to_csv("../reports/model_results_tensorflow.csv", index=False)
print("Đã lưu kết quả thành công vào ../reports/model_results_tensorflow.csv")

# Tải và tổng hợp cả 3 mô hình nếu có
dfs = []
for fname in ["model_results_numpy.csv", "model_results_pytorch.csv", "model_results_tensorflow.csv"]:
    p = os.path.join("../reports", fname)
    if os.path.exists(p):
        dfs.append(pd.read_csv(p))

if dfs:
    summary_df = pd.concat(dfs, ignore_index=True)
    summary_df.to_csv("../reports/model_results.csv", index=False)
    print("\n=== BẢNG TỔNG HỢP ĐỐI CHUẨN 3 FRAMEWORK TRÊN TẬP DIABETES ===")
    display(summary_df)


Đã lưu kết quả thành công vào ../reports/model_results_tensorflow.csv

=== BẢNG TỔNG HỢP ĐỐI CHUẨN 3 FRAMEWORK TRÊN TẬP DIABETES ===


,Framework,Dataset,Accuracy,Precision,Recall,F1,ROC_AUC,Train_Time_Sec,Epochs,Batch_Size
0,NumPy From Scratch,Diabetes (100k),0.970041,0.950161,0.696934,0.804082,0.972815,67.220477,12,256
1,PyTorch,Diabetes (100k),0.970145,0.957586,0.692217,0.803559,0.972953,24.557744,12,256
2,TensorFlow,Diabetes (100k),0.971029,0.993929,0.675708,0.804493,0.973306,5.724336,12,256
